# Derin Öğrenme ile Görüntü Sınıflandırma\nCNN ile CIFAR-10 veri seti üzerinde görüntü sınıflandırma.\n10 sınıf: uçak, araba, kuş, kedi, geyik, köpek, kurbağa, at, gemi, kamyon.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim\nimport torchvision, torchvision.transforms as T\nimport matplotlib.pyplot as plt, seaborn as sns, numpy as np, os\nfrom sklearn.metrics import confusion_matrix\nos.makedirs('cikti', exist_ok=True)\ndevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\nprint(f'Cihaz: {device}')

## Veri Seti: CIFAR-10

In [ ]:
transform = T.Compose([T.ToTensor(), T.Normalize((0.5,)*3, (0.5,)*3)])\n\ntrain_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)\ntest_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)\n\ntrain_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)\ntest_loader = torch.utils.data.DataLoader(test_set, batch_size=64, shuffle=False)\n\nsiniflar = ['ucak','araba','kus','kedi','geyik','kopek','kurbaga','at','gemi','kamyon']\nprint(f'Eğitim: {len(train_set)}, Test: {len(test_set)}')

In [ ]:
imgs, labels = next(iter(train_loader))\nfig, axes = plt.subplots(2, 5, figsize=(12, 5))\nfor i, ax in enumerate(axes.flat):\n    img = imgs[i].permute(1, 2, 0) * 0.5 + 0.5\n    ax.imshow(img); ax.set_title(siniflar[labels[i]]); ax.axis('off')\nplt.show()

## CNN Modeli

In [ ]:
class CNN(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.conv = nn.Sequential(\n            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d(2),\n            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d(2),\n            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(128), nn.MaxPool2d(2),\n        )\n        self.fc = nn.Sequential(\n            nn.Flatten(), nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.3),\n            nn.Linear(256, 10)\n        )\n    def forward(self, x): return self.fc(self.conv(x))\n\nmodel = CNN().to(device)\ntotal = sum(p.numel() for p in model.parameters())\nprint(f'Parametre: {total:,}')

## Eğitim

In [ ]:
criterion = nn.CrossEntropyLoss()\noptimizer = optim.Adam(model.parameters(), lr=0.001)\n\ntrain_losses, test_accs = [], []\nEPOCHS = 10\n\nfor epoch in range(EPOCHS):\n    model.train()\n    running_loss = 0\n    for imgs, labels in train_loader:\n        imgs, labels = imgs.to(device), labels.to(device)\n        optimizer.zero_grad()\n        loss = criterion(model(imgs), labels)\n        loss.backward()\n        optimizer.step()\n        running_loss += loss.item()\n    \n    model.eval()\n    correct, total = 0, 0\n    with torch.no_grad():\n        for imgs, labels in test_loader:\n            imgs, labels = imgs.to(device), labels.to(device)\n            preds = model(imgs).argmax(dim=1)\n            correct += (preds == labels).sum().item()\n            total += labels.size(0)\n    acc = 100*correct/total\n    train_losses.append(running_loss/len(train_loader))\n    test_accs.append(acc)\n    print(f'Epoch {epoch+1:2d}/{EPOCHS} | Loss: {train_losses[-1]:.4f} | Test Acc: {acc:.1f}%')

## Eğitim Grafikleri

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))\nax1.plot(train_losses); ax1.set_title('Eğitim Loss'); ax1.set_xlabel('Epoch')\nax2.plot(test_accs); ax2.set_title('Test Doğruluğu (%)'); ax2.set_xlabel('Epoch')\nplt.savefig('cikti/egitim.png', dpi=100, bbox_inches='tight')\nplt.show()

## Confusion Matrix

In [ ]:
all_preds, all_labels = [], []\nmodel.eval()\nwith torch.no_grad():\n    for imgs, labels in test_loader:\n        imgs = imgs.to(device)\n        preds = model(imgs).argmax(dim=1).cpu()\n        all_preds.extend(preds.tolist())\n        all_labels.extend(labels.tolist())\n\ncm = confusion_matrix(all_labels, all_preds)\nplt.figure(figsize=(10, 8))\nsns.heatmap(cm, annot=True, fmt='d', xticklabels=siniflar, yticklabels=siniflar, cmap='Blues')\nplt.xlabel('Tahmin'); plt.ylabel('Gerçek')\nplt.title('Confusion Matrix')\nplt.savefig('cikti/confusion_matrix.png', dpi=100, bbox_inches='tight')\nplt.show()\nprint('Eğitim tamamlandı.')